In [2]:
from langchain_core.documents import Document

In [3]:
doc=Document(page_content="This is a test document.",

 metadata={"source": "test.txt", "pages": 1,
 "author": "John Doe",
 "date_created": "2023-01-01",
 })
doc

Document(metadata={'source': 'test.txt', 'pages': 1, 'author': 'John Doe', 'date_created': '2023-01-01'}, page_content='This is a test document.')

In [4]:
#create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)


In [5]:
sample_text = {
    "file": "../data/text_files/python.txt",
    "content": "Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It has a large standard library and a vibrant ecosystem of third-party packages, making it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. Python's syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages."
}
with open(sample_text["file"], "w") as f:
    f.write(sample_text["content"])  
    #write mode will create the file if it doesn't exist and overwrite it if it does.


In [6]:
##Textloader

from langchain_community.document_loaders import TextLoader


loader = TextLoader("../data/text_files/python.txt")
documents = loader.load()
print(documents[0].page_content)
loader = TextLoader("../data/text_files/python.txt")
documents = loader.load()
print(documents)

c:\Users\Shrin\OneDrive\Desktop\rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It has a large standard library and a vibrant ecosystem of third-party packages, making it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. Python's syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages.
[Document(metadata={'source': '../data/text_files/python.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It has a large standard librar

In [7]:
##directory loader
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
  "../data/text_files", 
  glob="*.txt",
  loader_cls=TextLoader,
  loader_kwargs={"encoding": "utf-8"},
  show_progress=True)

documents = dir_loader.load()
print(documents)

100%|██████████| 2/2 [00:00<00:00, 164.24it/s]

[Document(metadata={'source': '..\\data\\text_files\\python.txt'}, page_content="Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming. It has a large standard library and a vibrant ecosystem of third-party packages, making it a popular choice for web development, data analysis, artificial intelligence, scientific computing, and more. Python's syntax emphasizes code readability, allowing developers to express concepts in fewer lines of code compared to other languages."), Document(metadata={'source': '..\\data\\text_files\\sample.txt'}, page_content='\n/data/text_files/python.txt/"Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python supports multiple pro

In [8]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

print("Current Directory:", os.getcwd())
print("PDF Folder Exists:", os.path.exists("../data/pdf_files"))
print("Files inside folder:", os.listdir("../data/pdf_files"))

dir_loader = DirectoryLoader(
    "../data/pdf_files",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents = dir_loader.load()

print(f"Total Documents Loaded: {len(pdf_documents)}")

Current Directory: c:\Users\Shrin\OneDrive\Desktop\rag\notebook
PDF Folder Exists: True
Files inside folder: ['Bagging and Boosting.pdf', 'sample_pdf.pdf']


100%|██████████| 2/2 [00:01<00:00,  1.75it/s]

Total Documents Loaded: 98


In [9]:
###embeddings
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
  def __init__(self,model_name:str ="all-MiniLM-L6-v2"):
    """
    Tnitilaze the embedding manager

    Args:
    model_name:HuggingFace model name for sentence embeddings
    """
    self.model_name=model_name
    self.model=None
    self._load_model()

  def _load_model(self):
    """Load the sentenceTransformer model"""  
    try:
      print(f"Loading embedding model:{self.model_name}")
      self.model=SentenceTransformer(self.model_name)
      print(f"Model loaded successsfully . Embedding dimensions:{self.model.get_sentence_embedding_dimension()}")
    except Exception as e:
      print(f"Error loading model{self.model_name}:{e}")  
      raise

  def generate_embeddings(self,texts: List[str])-> np.ndarray :
    """
    Generate embeddings for a list of texts

    Args:
    texts: List of text strings to embed

    Returns:
        numpy array of embeddings with shape(len(texts),embedding_dim)
    """ 
    if not self.model:
      raise ValueError("model not loaded")
    
    print(f"Generatng embedding for {len(texts)}texts..")
    embeddings= self.model.encode(texts,show_progress_bar=True)
    print(f"Generated embedding with shape:{embeddings.shape}")
    return embeddings
  

###initailize the embeddings manager
embeddings_manager=EmbeddingManager()
embeddings_manager  

Loading embedding model:all-MiniLM-L6-v2


c:\Users\Shrin\OneDrive\Desktop\rag\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded successsfully . Embedding dimensions:384


In [11]:
import os
import uuid
import chromadb
import numpy as np
from typing import List, Any


class VectorStore:
    def __init__(self,
                 collection_name: str = "pdf_documents",
                 persist_directory: str = "./data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["context_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document text
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to Chroma collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


# ✅ Create object (DO NOT overwrite class name)
vector_store = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 459


In [12]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [13]:
chunks=split_documents(pdf_documents)
chunks

Split 98 documents into 153 chunks

Example chunk:
Content: Generalized Linear Models 
  
  
Prerequisite:  
 Linear Regression 
 Logistic Regression 
Generalized Linear Models (GLMs) are a class of regression models that 
can be used to model a wide range...
Metadata: {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2025-05-20T01:06:32+05:30', 'source': '..\\data\\pdf_files\\Bagging and Boosting.pdf', 'file_path': '..\\data\\pdf_files\\Bagging and Boosting.pdf', 'total_pages': 95, 'format': 'PDF 1.5', 'title': '', 'author': 'Shriniwas Ande', 'subject': '', 'keywords': '', 'moddate': '2025-05-20T01:06:32+05:30', 'trapped': '', 'modDate': "D:20250520010632+05'30'", 'creationDate': "D:20250520010632+05'30'", 'page': 0}


[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2025-05-20T01:06:32+05:30', 'source': '..\\data\\pdf_files\\Bagging and Boosting.pdf', 'file_path': '..\\data\\pdf_files\\Bagging and Boosting.pdf', 'total_pages': 95, 'format': 'PDF 1.5', 'title': '', 'author': 'Shriniwas Ande', 'subject': '', 'keywords': '', 'moddate': '2025-05-20T01:06:32+05:30', 'trapped': '', 'modDate': "D:20250520010632+05'30'", 'creationDate': "D:20250520010632+05'30'", 'page': 0}, page_content='Generalized Linear Models \n\uf0b7  \n\uf0b7  \nPrerequisite:  \n\uf0b7 Linear Regression \n\uf0b7 Logistic Regression \nGeneralized Linear Models (GLMs) are a class of regression models that \ncan be used to model a wide range of relationships between a response \nvariable and one or more predictor variables. Unlike traditional linear \nregression models, which assume a linear relationship between the \nresponse and predictor variables, GLMs allow for more flexibl

In [14]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2025-05-20T01:06:32+05:30', 'source': '..\\data\\pdf_files\\Bagging and Boosting.pdf', 'file_path': '..\\data\\pdf_files\\Bagging and Boosting.pdf', 'total_pages': 95, 'format': 'PDF 1.5', 'title': '', 'author': 'Shriniwas Ande', 'subject': '', 'keywords': '', 'moddate': '2025-05-20T01:06:32+05:30', 'trapped': '', 'modDate': "D:20250520010632+05'30'", 'creationDate': "D:20250520010632+05'30'", 'page': 0}, page_content='Generalized Linear Models \n\uf0b7  \n\uf0b7  \nPrerequisite:  \n\uf0b7 Linear Regression \n\uf0b7 Logistic Regression \nGeneralized Linear Models (GLMs) are a class of regression models that \ncan be used to model a wide range of relationships between a response \nvariable and one or more predictor variables. Unlike traditional linear \nregression models, which assume a linear relationship between the \nresponse and predictor variables, GLMs allow for more flexibl

In [15]:
### convert the text to embeddings
text=[doc.page_content for doc in chunks]

##genrate the embeddings
embeddings=embeddings_manager.generate_embeddings(text)

#store int he vector database

vector_store.add_document(chunks, embeddings)

Generatng embedding for 153texts..


Batches: 100%|██████████| 5/5 [00:10<00:00,  2.15s/it]


Generated embedding with shape:(153, 384)
Adding 153 documents to vector store
Successfully added 153 documents
Total documents in collection: 612


In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embeddings_manager)

In [17]:


rag_retriever

In [19]:
# Define a query
query = "What is Python used for?"
 
# Retrieve relevant documents
retrieved_docs = rag_retriever.retrieve(query, top_k=5, score_threshold=0.5)
print("Retrieved Documents:", retrieved_docs)
 
# Format the retrieved context
context = "\n\n".join([doc['content'] for doc in retrieved_docs])
print("Context for generation:", context)
 
# Generate a response using a language model
from transformers import pipeline
 
# Load a pre-trained language model pipeline
generator = pipeline("text-generation", model="gpt2")
 
# Combine query and context
prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
response = generator(prompt, max_length=200, num_return_sequences=1)
print("Generated Response:", response[0]['generated_text'])

Retrieving documents for query: 'What is Python used for?'
Top K: 5, Score threshold: 0.5
Generatng embedding for 1texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.68it/s]

Generated embedding with shape:(1, 384)
Retrieved 0 documents (after filtering)
Retrieved Documents: []
Context for generation: 



c:\Users\Shrin\OneDrive\Desktop\rag\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shrin\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to expl

Generated Response: Context:


Question: What is Python used for?

Answer: Python runs as a normal application. This is good and you can get away with it if your programming language is a bit different than Java - because Java is a special case - but we are using Python right now.

Question: So what if I just read this and just want to run my code in Python?

Answer: You could do this with the -python switch in the standard IDE (e.g. by using the -h switch).

Question: That's really nice? What would the first switch be like?

Answer: -python would automatically switch the file to a directory where the Python program will run. You can use any of several directories, or a whole bunch of directories and file types.


The Python standard library comes with two things that will help you choose just that specific file. This is a "command-line" program which requires some configuration and even


In [22]:
# Define a query related to bagging and boosting
query = "What is the difference between bagging and boosting in machine learning?"
 
# Retrieve relevant documents
retrieved_docs = rag_retriever.retrieve(query, top_k=5, score_threshold=0.5)
print("Retrieved Documents:", retrieved_docs)
 
# Format the retrieved context
context = "\n\n".join([doc['content'] for doc in retrieved_docs])
print("Context for generation:", context)
 
# Generate a response using a language model
from transformers import pipeline
 
# Load a pre-trained language model pipeline
generator = pipeline("text-generation", model="gpt2")
 
# Combine query and context
prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
response = generator(prompt, max_length=100, num_return_sequences=1)
print("Generated Response:", response[0]['generated_text'])

Retrieving documents for query: 'What is the difference between bagging and boosting in machine learning?'
Top K: 5, Score threshold: 0.5
Generatng embedding for 1texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.69it/s]

Generated embedding with shape:(1, 384)
Retrieved 5 documents (after filtering)
Retrieved Documents: [{'id': 'doc_fe16ddfd_38', 'content': 'Definition: Bagging is an ensemble technique that builds multiple models \nindependently using random subsets of the data (created via \nbootstrapping), and combines their outputs. \nSteps: \n1. Generate multiple bootstrapped datasets from the original data. \n2. Train a separate model on each bootstrapped dataset. \n3. For classification: use majority voting. \n4. For regression: use average of predictions. \nExample: Random Forest is a classic example of Bagging using decision trees. \nImpact on Bias and Variance: \n\uf0b7 Reduces Variance. \n\uf0b7 Bias remains the same or slightly reduced. \n\uf0b7 Helps in overfitting-prone models like decision trees. \n \n3. Boosting:', 'metadata': {'title': '', 'total_pages': 95, 'author': 'Shriniwas Ande', 'creationDate': "D:20250520010632+05'30'", 'creator': 'Microsoft® Word 2010', 'context_length': 653, '


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


ValueError: Input length of input_ids is 100, but `max_length` is set to 100. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.